Install and import packages

In [1]:
!pip -q install datasets pandas numpy scikit-learn

In [2]:
import itertools
import numpy as np
import pandas as pd

from datasets import load_dataset
from scipy.sparse import csr_matrix

from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

Load the dataset

In [3]:
dataset = load_dataset("tattabio/ec_classification")

train_df = pd.DataFrame(dataset["train"])
test_df = pd.DataFrame(dataset["test"])

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

train_df.head()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/485 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/285k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/87.5k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/512 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/128 [00:00<?, ? examples/s]

Train shape: (512, 3)
Test shape: (128, 3)


,Entry,Label,Sequence
0,Q9LQC0,1.14.14.18,MATSRLNASCRFPASRRLDCESYVSLRAKTVTIRYVRTIAAPRRHL...
1,A0AFT8,1.14.14.18,MIIVTNTIKVEKGAAEHVIRQFTGANGDGHPTKDIAEVEGFLGFEL...
2,P74133,1.14.14.18,MTNLAQKLRYGTQQSHTLAENTAYMKCFLKGIVEREPFRQLLANLY...
3,O31534,1.14.14.18,MFVQLRKMTVKEGFADKVIERFSAEGIIEKQEGLIDVTVLEKNVRR...
4,P34697,1.15.1.1,MFMNLLTQVSNAIFPQVEAAQKMSNRAVAVLRGETVTGTIWITQKS...



Inspect the labels
---



In [4]:
print("Number of unique labels in train:", train_df["Label"].nunique())
print("Number of unique labels in test:", test_df["Label"].nunique())

print("\nTop label counts in train:")
print(train_df["Label"].value_counts().head(10))

Number of unique labels in train: 128
Number of unique labels in test: 128

Top label counts in train:
Label
1.14.14.18    4
1.15.1.1      4
1.16.3.1      4
1.17.4.1      4
1.5.98.3      4
1.8.3.2       4
2.1.1.354     4
2.1.1.360     4
2.1.1.37      4
2.1.1.72      4
Name: count, dtype: int64


Define 3-mer features

In [5]:
AMINO_ACIDS = list("ACDEFGHIKLMNPQRSTVWY")
VALID_AA_SET = set(AMINO_ACIDS)

K = 3
KMERS = [''.join(p) for p in itertools.product(AMINO_ACIDS, repeat=K)]
KMER_TO_INDEX = {kmer: i for i, kmer in enumerate(KMERS)}

print("Number of 3-mers:", len(KMERS))
print("First 20 3-mers:", KMERS[:20])

Number of 3-mers: 8000
First 20 3-mers: ['AAA', 'AAC', 'AAD', 'AAE', 'AAF', 'AAG', 'AAH', 'AAI', 'AAK', 'AAL', 'AAM', 'AAN', 'AAP', 'AAQ', 'AAR', 'AAS', 'AAT', 'AAV', 'AAW', 'AAY']


Build sparse 3-mer feature matrix

In [6]:
def build_kmer_feature_matrix_sparse(sequences, k=3):
    """
    Inputs
    ----------
    sequences : iterable of str
        Protein sequences
    k : int
        k-mer length

    Returns
    -------
    scipy.sparse.csr_matrix
        Sparse matrix of shape (n_samples, 20^k)
    """
    rows = []
    cols = []
    data = []

    for row_idx, seq in enumerate(sequences):
        seq = str(seq).upper()
        counts = {}
        valid_count = 0

        for i in range(len(seq) - k + 1):
            kmer = seq[i:i+k]
            if all(ch in VALID_AA_SET for ch in kmer):
                col_idx = KMER_TO_INDEX[kmer]
                counts[col_idx] = counts.get(col_idx, 0) + 1
                valid_count += 1

        if valid_count > 0:
            for col_idx, count in counts.items():
                rows.append(row_idx)
                cols.append(col_idx)
                data.append(count / valid_count)

    X_sparse = csr_matrix((data, (rows, cols)), shape=(len(sequences), len(KMERS)), dtype=np.float32)
    return X_sparse

buildx and y


In [7]:
X_train = build_kmer_feature_matrix_sparse(train_df["Sequence"], k=3)
X_test = build_kmer_feature_matrix_sparse(test_df["Sequence"], k=3)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("X_train type:", type(X_train))

X_train shape: (512, 8000)
X_test shape: (128, 8000)
X_train type: <class 'scipy.sparse._csr.csr_matrix'>


In [8]:
label_encoder = LabelEncoder()

y_train = label_encoder.fit_transform(train_df["Label"])
y_test = label_encoder.transform(test_df["Label"])

print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)
print("Number of classes:", len(label_encoder.classes_))

y_train shape: (512,)
y_test shape: (128,)
Number of classes: 128


Quick sanity check

In [9]:
print("Number of nonzero entries in X_train:", X_train.nnz)
print("Average nonzero features per training sample:", X_train.nnz / X_train.shape[0])

Number of nonzero entries in X_train: 241892
Average nonzero features per training sample: 472.4453125


In [10]:
example_row = X_train[0].toarray().flatten()
nonzero_indices = np.nonzero(example_row)[0]

print("Number of nonzero 3-mers in first sequence:", len(nonzero_indices))
print("First 10 nonzero 3-mers:")
for idx in nonzero_indices[:10]:
    print(KMERS[idx], "->", example_row[idx])

Number of nonzero 3-mers in first sequence: 275
First 10 nonzero 3-mers:
AAA -> 0.0035587188
AAG -> 0.0035587188
AAP -> 0.0035587188
AEK -> 0.0071174377
AEL -> 0.0035587188
AEN -> 0.0035587188
AFI -> 0.0035587188
AGE -> 0.0035587188
AGG -> 0.0035587188
AGL -> 0.0035587188


Train logistic regression model

In [11]:
model = LogisticRegression(
    C=1.0,
    max_iter=3000,
    solver="saga",
    class_weight="balanced",
    multi_class="auto",
    n_jobs=-1
)

model.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


LogisticRegression(class_weight='balanced', max_iter=3000, multi_class='auto',
                   n_jobs=-1, solver='saga')

make prediction

In [12]:
y_pred = model.predict(X_test)

evaluate the model

In [13]:
accuracy = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")
weighted_f1 = f1_score(y_test, y_pred, average="weighted")

print(f"Accuracy: {accuracy:.4f}")
print(f"Macro F1: {macro_f1:.4f}")
print(f"Weighted F1: {weighted_f1:.4f}")

Accuracy: 0.0781
Macro F1: 0.0588
Weighted F1: 0.0588


In [14]:
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

Classification Report:
              precision    recall  f1-score   support

  1.14.14.18       0.00      0.00      0.00         1
    1.15.1.1       0.00      0.00      0.00         1
    1.16.3.1       0.00      0.00      0.00         1
    1.17.4.1       0.00      0.00      0.00         1
    1.5.98.3       0.00      0.00      0.00         1
     1.8.3.2       0.00      0.00      0.00         1
   2.1.1.354       0.00      0.00      0.00         1
   2.1.1.360       0.00      0.00      0.00         1
    2.1.1.37       0.00      0.00      0.00         1
    2.1.1.72       0.00      0.00      0.00         1
    2.1.1.77       1.00      1.00      1.00         1
    2.1.1.86       0.00      0.00      0.00         1
    2.1.3.15       0.00      0.00      0.00         1
   2.3.1.225       0.00      0.00      0.00         1
   2.3.1.269       0.00      0.00      0.00         1
    2.3.1.48       0.00      0.00      0.00         1
    2.3.1.51       0.00      0.00      0.00         1
    

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


compare with baseline

In [15]:
majority_label = train_df["Label"].mode()[0]
baseline_acc = (test_df["Label"] == majority_label).mean()

print("Majority class in training set:", majority_label)
print(f"Majority-class baseline accuracy: {baseline_acc:.4f}")

Majority class in training set: 1.14.14.18
Majority-class baseline accuracy: 0.0078


Show predictions alongside true labels

In [16]:
pred_labels = label_encoder.inverse_transform(y_pred)
true_labels = label_encoder.inverse_transform(y_test)

results_df = pd.DataFrame({
    "Entry": test_df["Entry"],
    "True_Label": true_labels,
    "Predicted_Label": pred_labels,
    "Correct": true_labels == pred_labels
})

results_df.head(20)

,Entry,True_Label,Predicted_Label,Correct
0,A0A0H2XEA6,1.14.14.18,6.1.1.19,False
1,Q5AD07,1.15.1.1,4.2.2.2,False
2,O74831,1.16.3.1,3.1.21.10,False
3,O46310,1.17.4.1,6.1.1.19,False
4,Q8PU58,1.5.98.3,3.1.4.12,False
5,Q5UQV6,1.8.3.2,6.1.1.19,False
6,Q8IRW8,2.1.1.354,3.6.1.23,False
7,Q6BTC8,2.1.1.360,3.1.21.10,False
8,P0DOY1,2.1.1.37,6.1.1.19,False
9,Q09956,2.1.1.72,3.1.21.10,False


In [17]:
print("Number correct:", results_df["Correct"].sum())
print("Number incorrect:", (~results_df["Correct"]).sum())

Number correct: 10
Number incorrect: 118


find the best hyperparametes

In [18]:
hyperparameter_results = []

for C in [0.1, 1.0, 10.0]:
    for max_iter in [2000, 3000, 5000]:
        clf = LogisticRegression(
            C=C,
            max_iter=max_iter,
            solver="saga",
            class_weight="balanced",
            multi_class="auto",
            n_jobs=-1
        )
        clf.fit(X_train, y_train)
        preds = clf.predict(X_test)

        hyperparameter_results.append({
            "C": C,
            "max_iter": max_iter,
            "accuracy": accuracy_score(y_test, preds),
            "macro_f1": f1_score(y_test, preds, average="macro"),
            "weighted_f1": f1_score(y_test, preds, average="weighted")
        })

hyperparameter_df = pd.DataFrame(hyperparameter_results)
hyperparameter_df.sort_values(by=["macro_f1", "accuracy"], ascending=False).reset_index(drop=True)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and wi

,C,max_iter,accuracy,macro_f1,weighted_f1
0,10.0,5000,0.179688,0.144959,0.144959
1,10.0,3000,0.187500,0.141146,0.141146
2,10.0,2000,0.156250,0.123847,0.123847
3,1.0,5000,0.054688,0.039583,0.039583
4,1.0,3000,0.062500,0.037677,0.037677
5,1.0,2000,0.062500,0.036001,0.036001
6,0.1,3000,0.023438,0.015748,0.015748
7,0.1,5000,0.023438,0.015748,0.015748
8,0.1,2000,0.015625,0.005331,0.005331


In [19]:
best_row = hyperparameter_df.sort_values(
    by=["macro_f1", "accuracy"], ascending=False
).iloc[0]

print(best_row)

C                10.000000
max_iter       5000.000000
accuracy          0.179688
macro_f1          0.144959
weighted_f1       0.144959
Name: 8, dtype: float64


train the model with the best hyperparameters

In [20]:
best_model = LogisticRegression(
    C=float(best_row["C"]),
    max_iter=int(best_row["max_iter"]),
    solver="saga",
    class_weight="balanced",
    multi_class="auto",
    n_jobs=-1
)

best_model.fit(X_train, y_train)
best_pred = best_model.predict(X_test)

print("Best 3-mer model accuracy:", accuracy_score(y_test, best_pred))
print("Best 3-mer model macro F1:", f1_score(y_test, best_pred, average="macro"))
print("Best 3-mer model weighted F1:", f1_score(y_test, best_pred, average="weighted"))

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Best 3-mer model accuracy: 0.1875
Best 3-mer model macro F1: 0.147718253968254
Best 3-mer model weighted F1: 0.147718253968254
